### Project Goal
#### Load Data → Clean Data → Explore Price Trends → Build Interactive Charts → Export for Portfolio

### Cell 1 — Project setup

In [ ]:
import sys
from pathlib import Path

# Resolve the project root whether this notebook is run from the repo
# root or from inside the notebooks/ folder (portable across machines).
PROJECT_PATH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_PATH = str(PROJECT_PATH)

if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)

DATA_PATH = Path(PROJECT_PATH) / "data" / "incoming" / "BitCoin_Historical_Data_2025.csv"
CHARTS_DIR = Path(PROJECT_PATH) / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project path:", PROJECT_PATH)

### Cell 2 — Load raw Bitcoin dataset

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

display(df.head())
print("Raw dataset shape:", df.shape)

### Cell 3 — Check data types, missing values, duplicates

In [ ]:
print("Data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

### Cell 4 — Clean and prepare the dataset

In [ ]:
# Keep only the columns needed for price analysis, parse the timestamp,
# and sort chronologically (source data is oldest-to-newest already, but
# we sort explicitly so the notebook doesn't depend on that assumption).
data = df[["Open time", "Open", "High", "Low", "Close", "Volume"]].copy()
data = data.rename(columns={"Open time": "Date"})
data["Date"] = pd.to_datetime(data["Date"])
data = data.sort_values("Date").reset_index(drop=True)

display(data.head())
print("Date range:", data["Date"].min(), "to", data["Date"].max())
print("Cleaned dataset shape:", data.shape)

### Cell 5 — Create visualization helper (dark theme)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def apply_dark_theme(fig, title):
    fig.update_layout(
        title={
            "text": title,
            "x": 0.05,
            "xanchor": "left",
            "font": {"size": 22, "color": "white"}
        },
        paper_bgcolor="#0b0f19",
        plot_bgcolor="#0b0f19",
        font={"family": "Arial", "color": "white", "size": 13},
        xaxis=dict(showgrid=True, gridcolor="#22304a", zeroline=False, showline=False),
        yaxis=dict(showgrid=True, gridcolor="#22304a", zeroline=False, showline=False),
        margin=dict(l=60, r=40, t=70, b=60),
        height=600,
        width=1000
    )
    return fig

### Cell 6 — Price overview chart (Open / High / Low / Close)

In [ ]:
fig_overview = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Open", "High", "Low", "Close")
)

fig_overview.add_trace(go.Scatter(x=data["Date"], y=data["Open"], name="Open", line=dict(color="#4FC3F7")), row=1, col=1)
fig_overview.add_trace(go.Scatter(x=data["Date"], y=data["High"], name="High", line=dict(color="#66BB6A")), row=1, col=2)
fig_overview.add_trace(go.Scatter(x=data["Date"], y=data["Low"], name="Low", line=dict(color="#EF5350")), row=2, col=1)
fig_overview.add_trace(go.Scatter(x=data["Date"], y=data["Close"], name="Close", line=dict(color="#FF9800")), row=2, col=2)

apply_dark_theme(fig_overview, "Bitcoin Price Overview — 2025")
fig_overview.update_layout(showlegend=False, height=700)
fig_overview.show()

### Cell 7 — Candlestick chart

In [ ]:
trace = go.Candlestick(
    x=data["Date"],
    open=data["Open"],
    high=data["High"],
    low=data["Low"],
    close=data["Close"],
    increasing_line_color="#26A69A",
    decreasing_line_color="#EF5350"
)

fig_candlestick = go.Figure(data=[trace])
apply_dark_theme(fig_candlestick, "Bitcoin Daily Candlestick — 2025")
fig_candlestick.update_layout(
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    xaxis_rangeslider_visible=False,
    height=650
)
fig_candlestick.show()

### Cell 8 — Linear vs. log scale comparison

In [ ]:
fig_scale = make_subplots(rows=1, cols=2, subplot_titles=("Linear scale", "Log scale"))

fig_scale.add_trace(
    go.Scatter(x=data["Date"], y=data["Close"], mode="lines", name="Close", line=dict(color="#4FC3F7")),
    row=1, col=1
)
fig_scale.add_trace(
    go.Scatter(x=data["Date"], y=data["Close"], mode="lines", name="Close", line=dict(color="#4FC3F7"), showlegend=False),
    row=1, col=2
)

fig_scale.update_yaxes(type="linear", title_text="Price (USD)", row=1, col=1)
fig_scale.update_yaxes(type="log", title_text="Price (USD, log)", row=1, col=2)

apply_dark_theme(fig_scale, "Bitcoin Close Price — Linear vs. Log Scale")
fig_scale.update_layout(showlegend=False, height=550)
fig_scale.show()

### Cell 9 — Monthly average close price

In [ ]:
monthly_close = (
    data.set_index("Date")["Close"]
    .resample("ME")
    .mean()
    .reset_index()
)

fig_monthly = go.Figure(
    go.Bar(x=monthly_close["Date"], y=monthly_close["Close"], marker_color="#7C4DFF")
)
apply_dark_theme(fig_monthly, "Bitcoin Average Close Price by Month — 2025")
fig_monthly.update_layout(xaxis_title="Month", yaxis_title="Avg. Close Price (USD)")
fig_monthly.show()

### Cell 10 — Daily percentage change in closing price

In [ ]:
data["close_pct_change"] = data["Close"].pct_change() * 100

fig_pct_change = go.Figure(
    go.Scatter(x=data["Date"], y=data["close_pct_change"], mode="lines", line=dict(color="#FF4081", width=1))
)
apply_dark_theme(fig_pct_change, "Bitcoin Daily % Change in Closing Price — 2025")
fig_pct_change.update_layout(xaxis_title="Date", yaxis_title="Daily change (%)")
fig_pct_change.show()

print("Largest single-day gain: {:.2f}% on {}".format(
    data['close_pct_change'].max(), data.loc[data['close_pct_change'].idxmax(), 'Date'].date()
))
print("Largest single-day drop: {:.2f}% on {}".format(
    data['close_pct_change'].min(), data.loc[data['close_pct_change'].idxmin(), 'Date'].date()
))

### Cell 11 — Export charts for the portfolio website

In [ ]:
fig_overview.write_html(CHARTS_DIR / "bitcoin_price_overview.html")
fig_candlestick.write_html(CHARTS_DIR / "bitcoin_candlestick.html")
fig_scale.write_html(CHARTS_DIR / "bitcoin_linear_vs_log_scale.html")
fig_monthly.write_html(CHARTS_DIR / "bitcoin_monthly_avg_close.html")
fig_pct_change.write_html(CHARTS_DIR / "bitcoin_daily_pct_change.html")

print("All 5 charts exported to:", CHARTS_DIR)

### Summary

This notebook cleans a full year (2025) of daily Bitcoin OHLCV price data, explores price
trends through an overview chart, a candlestick chart, a linear-vs-log scale comparison, a
monthly average price view, and a daily percent-change view, then exports all five charts as
standalone interactive HTML files used on the Bitcoin case study page.

This replaces an earlier practice/training notebook that had been run against placeholder
data unrelated to real Bitcoin prices — all analysis here uses the real 2025 BTC-USD daily
series.